In [35]:
import pandas as pd
from sklearn.model_selection import train_test_split
# Charger le dataset
df = pd.read_csv('Clean_Dataset.csv')
df

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955
...,...,...,...,...,...,...,...,...,...,...,...,...
300148,300148,Vistara,UK-822,Chennai,Morning,one,Evening,Hyderabad,Business,10.08,49,69265
300149,300149,Vistara,UK-826,Chennai,Afternoon,one,Night,Hyderabad,Business,10.42,49,77105
300150,300150,Vistara,UK-832,Chennai,Early_Morning,one,Night,Hyderabad,Business,13.83,49,79099
300151,300151,Vistara,UK-828,Chennai,Early_Morning,one,Evening,Hyderabad,Business,10.00,49,81585


In [36]:
from sklearn.preprocessing import StandardScaler

# 1. Nettoyage : On ne garde que les colonnes utiles
df = df.drop(["flight", "Unnamed: 0"], axis=1)

# 2. Séparation : On isole la cible (Price)
X = df.drop("price", axis=1)
y = df["price"]

# 3. Encodage Ordinal : On transforme le texte en nombres logiques
X['stops'] = X['stops'].map({'zero': 0, 'one': 1, 'two_or_more': 2})
X['class'] = X['class'].map({'Economy': 0, 'Business': 1})

# 4. Encodage Nominal : On crée les colonnes 0/1 pour le reste
nominale_cols = ['airline', 'departure_time', 'arrival_time', 'source_city', 'destination_city']
X = pd.get_dummies(X, columns=nominale_cols, dtype=int)

# 5. Scaling : On normalise les colonnes numériques
scaler = StandardScaler()
X[["duration", "days_left"]] = scaler.fit_transform(X[["duration", "days_left"]])

# Affichage du résultat final
print("Data prête pour le modèle :")
display(X.head())


Data prête pour le modèle :


,stops,class,duration,days_left,airline_AirAsia,airline_Air_India,airline_GO_FIRST,airline_Indigo,airline_SpiceJet,airline_Vistara,...,source_city_Delhi,source_city_Hyderabad,source_city_Kolkata,source_city_Mumbai,destination_city_Bangalore,destination_city_Chennai,destination_city_Delhi,destination_city_Hyderabad,destination_city_Kolkata,destination_city_Mumbai
0,0,0,-1.397531,-1.843875,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,1
1,0,0,-1.375284,-1.843875,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,1
2,0,0,-1.397531,-1.843875,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,1
3,0,0,-1.386407,-1.843875,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,1
4,0,0,-1.375284,-1.843875,0,0,0,0,0,1,...,1,0,0,0,0,0,0,0,0,1


In [4]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Initialisation de la liste pour stocker les scores
results = []

# --- ÉTAPE 1 : LA DIVISION (SPLIT) ---
# On sépare les données en "Entraînement" (80%) et "Test" (20%)
# random_state=42 permet de figer le hasard pour avoir toujours les mêmes résultats
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

# --- ÉTAPE 2 : CHOIX ET APPRENTISSAGE ---
model = LinearRegression()

# .fit() : C'est ici que le modèle "apprend". 
# Il cherche la relation mathématique entre X_train et y_train.
model.fit(X_train, y_train)

# --- ÉTAPE 3 : LA PRÉDICTION ---
# .predict() : On donne au modèle des données qu'il n'a jamais vues (X_test).
# Il nous rend ses "estimations" (y_pred).
y_pred = model.predict(X_test)

# --- ÉTAPE 4 : LE CALCUL DES ERREURS ---
# On compare les vraies valeurs (y_test) avec les prédictions (y_pred)

# MAE : L'erreur moyenne absolue. Facile à lire (ex: on se trompe de 5€ en moyenne).
mae = mean_absolute_error(y_test, y_pred)

# RMSE : Donne plus de poids aux grosses erreurs. Si le RMSE est beaucoup plus haut 
# que la MAE, cela veut dire que tu as des prédictions très très éloignées de la réalité.
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# R² : Score de 0 à 1. Plus c'est proche de 1, plus le modèle est performant.
r2 = r2_score(y_test, y_pred)

# --- ÉTAPE 5 : LE STOCKAGE ET L'AFFICHAGE ---
results.append({
    "Modèle": "Linear Regression",
    "MAE": round(mae, 2),
    "RMSE": round(rmse, 2),
    "R² Test": round(r2, 3)
})

df_results = pd.DataFrame(results)
print(df_results.sort_values(by="R² Test", ascending=False))

              Modèle      MAE     RMSE  R² Test
0  Linear Regression  4500.71  6814.94     0.91


In [40]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

# =========================
# 1. SPLIT DES DONNÉES
# =========================
# On garde y (Price) en valeur réelle

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# =========================
# 2. CRÉATION DU MODÈLE - GRADIENT BOOSTING
# =========================
# Gradient Boosting: PLUS RAPIDE et MEILLEUR que RandomForest
model = GradientBoostingRegressor(
    n_estimators=150,        # Rapide mais précis
    learning_rate=0.05,      # Réduit progressivement l'erreur
    max_depth=5,             # Arbres simples = meilleure généralisation
    min_samples_split=5,
    min_samples_leaf=2,
    subsample=0.8,           # Utilise 80% des données
    random_state=42
)

# =========================
# 3. ENTRAÎNEMENT
# =========================
model.fit(X_train, y_train)

print("✓ Modèle Gradient Boosting entraîné rapidement!")

# =========================
# 4. PRÉDICTION
# =========================
y_pred = model.predict(X_test)

print("Quelques prédictions :", y_pred[:5])

# =========================
# 5. ÉVALUATION (RÉELLE - MAE SANS SCALING)
# =========================
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n📊 RÉSULTATS (PRIX RÉELS) :")
print(f"➡️ MAE (Erreur moyenne) : {mae:.2f} €")
print(f"➡️ R² (Précision) : {r2 * 100:.2f} %")

# =========================
# 6. COMPARAISON (OPTIONNEL)
# =========================
# Voir vrai vs prédiction

for i in range(5):
    print(f"Vrai prix: {y_test.iloc[i]} € | Prédit: {y_pred[i]:.2f} €")

✓ Modèle Gradient Boosting entraîné rapidement!
Quelques prédictions : [ 5501.76989259 67279.44006724  7326.32981621 56706.45915366
  5628.35548842]

📊 RÉSULTATS (PRIX RÉELS) :
➡️ MAE (Erreur moyenne) : 2571.34 €
➡️ R² (Précision) : 96.20 %
Vrai prix: 7366 € | Prédit: 5501.77 €
Vrai prix: 64831 € | Prédit: 67279.44 €
Vrai prix: 6195 € | Prédit: 7326.33 €
Vrai prix: 60160 € | Prédit: 56706.46 €
Vrai prix: 6578 € | Prédit: 5628.36 €


In [41]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score





# =========================
# 4. FEATURE ENGINEERING 🔥
# =========================
X["duration_per_stop"] = X["duration"] / (X["stops"] + 1)
X["duration_class"] = X["duration"] * X["class"]
X["urgency"] = 1 / (X["days_left"] + 1)



# =========================
# 6. SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# 7. GRID SEARCH 🔥
# =========================
param_grid = {
    'n_estimators': [300, 400],
    'max_depth': [20, 30, None]
    # 'min_samples_split': [2, 5],
    # 'min_samples_leaf': [1, 2]
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

grid = GridSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring='neg_mean_absolute_error',
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Meilleurs paramètres :", grid.best_params_)

# =========================
# 8. PRÉDICTION
# =========================
y_pred = best_model.predict(X_test)

# =========================
# 9. ÉVALUATION
# =========================
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n📊 RÉSULTATS RANDOM FOREST :")
print(f"➡️ MAE : {mae:.2f} €")
print(f"➡️ R² : {r2 * 100:.2f} %")

KeyboardInterrupt: 

In [ ]:
from sklearn.ensemble import  RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
import pickle
import json

# =========================
# 7. MODÉLISATION - COMPARAISON DE MODÈLES
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Définition des modèles
models = {
    'DummyRegressor': DummyRegressor(strategy='mean'),
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
}

# Évaluation avec cross-validation
results_cv = []

for name, model in models.items():
    # Cross-validation (MAE négatif par défaut, on inverse)
    cv_scores = -cross_val_score(model, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
    
    results_cv.append({
        'Modèle': name,
        'MAE CV': cv_scores.mean(),
        'Std CV': cv_scores.std()
    })
    
    print(f"{name}: MAE = {cv_scores.mean():.2f} € (±{cv_scores.std():.2f})")

print("\n✓ Cross-validation terminée!")

# =========================
# 8. ÉVALUATION FINALE - MEILLEUR MODÈLE
# =========================
# Trouver le modèle avec le meilleur MAE CV
df_results = pd.DataFrame(results_cv)
best_model_name = df_results.loc[df_results['MAE CV'].idxmin(), 'Modèle']
print(f"🏆 Meilleur modèle sélectionné: {best_model_name}")

# Réentraîner le meilleur modèle sur données complètes
best_model = models[best_model_name]
best_model.fit(X_train, y_train)

# Prédictions
y_pred = best_model.predict(X_test)

# Métriques finales
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"\n📊 RÉSULTATS FINAUX - {best_model_name} :")
print(f"MAE : {mae:.2f} €")
print(f"RMSE : {rmse:.2f} €")
print(f"R² : {r2 * 100:.2f} %")

# =========================
# 9. INTERVALLE DE CONFIANCE (95%)
# =========================
erreurs_absolues = np.abs(y_test - y_pred)
mae_std = erreurs_absolues.std()
erreur_standard = mae_std / np.sqrt(len(y_test))

# Intervalle de confiance à 95%
z_score = 1.96  # Pour 95%
ic_inf = mae - (z_score * erreur_standard)
ic_sup = mae + (z_score * erreur_standard)

print(f"\n📈 INTERVALLE DE CONFIANCE (95%) :")
print(f"MAE estimée : {mae:.2f} €")
print(f"Intervalle : [{ic_inf:.2f} €, {ic_sup:.2f} €]")

# =========================
# 10. EXPORT DU MODÈLE
# =========================


# Sauvegarder le modèle
with open('flight_price_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# Sauvegarder le scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Métadonnées
metadata = {
    'model_name': 'Gradient Boosting Regressor',
    'mae': mae,
    'rmse': rmse,
    'r2': r2,
    'ic_95': [ic_inf, ic_sup],
    'features': X.columns.tolist()
}


with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ MODÈLE SAUVEGARDÉ :")
print(f"  - flight_price_model.pkl")
print(f"  - scaler.pkl")
print(f"  - model_metadata.json")

DummyRegressor: MAE = 19757.19 € (±19.59)
Linear Regression: MAE = 4512.29 € (±15.82)
Ridge Regression: MAE = 4512.26 € (±15.82)
Random Forest: MAE = 1110.95 € (±8.53)
Gradient Boosting: MAE = 2927.58 € (±9.51)

✓ Cross-validation terminée!
🏆 Meilleur modèle sélectionné: Random Forest

📊 RÉSULTATS FINAUX - Random Forest :
MAE : 1080.68 €
RMSE : 2768.90 €
R² : 98.51 %

📈 INTERVALLE DE CONFIANCE (95%) :
MAE estimée : 1080.68 €
Intervalle : [1060.28 €, 1101.07 €]

✅ MODÈLE SAUVEGARDÉ :
  - flight_price_model.pkl
  - scaler.pkl
  - model_metadata.json
